In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime, timezone

# Load CSVs (they are in the current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Ensure timestamp is integer
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# Helper: convert epoch seconds (assumed UTC) to Asia/Shanghai tz-aware ISO string
tz_sh = pytz.timezone('Asia/Shanghai')
def epoch_to_sh_string(s):
    # s can be pandas scalar (int)
    if pd.isna(s):
        return None
    # treat s as UTC epoch seconds, produce Asia/Shanghai time string
    dt_utc = datetime.fromtimestamp(int(s), tz=timezone.utc)
    dt_sh = dt_utc.astimezone(tz_sh)
    return dt_sh.isoformat()

# Global summaries for metric.csv
metric_total = int(metric_df.shape[0])
metric_ts_min = epoch_to_sh_string(metric_df['timestamp'].min()) if metric_total>0 else None
metric_ts_max = epoch_to_sh_string(metric_df['timestamp'].max()) if metric_total>0 else None
metric_unique_cmdb = metric_df['cmdb_id'].dropna().unique().tolist()

metric_top_kpis = (
    metric_df['kpi_name']
    .value_counts()
    .head(20)
    .reset_index()
    .rename(columns={'index':'kpi_name','kpi_name':'count'})
)
metric_top_kpis_list = metric_top_kpis.to_dict(orient='records')

# Global summaries for trace.csv
trace_total = int(trace_df.shape[0])
trace_ts_min = epoch_to_sh_string(trace_df['timestamp'].min()) if trace_total>0 else None
trace_ts_max = epoch_to_sh_string(trace_df['timestamp'].max()) if trace_total>0 else None
trace_unique_cmdb = trace_df['cmdb_id'].dropna().unique().tolist()

trace_top_names = (
    trace_df['trace_name']
    .value_counts()
    .head(20)
    .reset_index()
    .rename(columns={'index':'trace_name','trace_name':'count'})
)
trace_top_names_list = trace_top_names.to_dict(orient='records')

# Incident time range: May 23, 2020 00:30 to 01:00 in UTC+8 (Asia/Shanghai)
start_sh = tz_sh.localize(datetime(2020,5,23,0,30,0))
end_sh = tz_sh.localize(datetime(2020,5,23,1,0,0))
# Convert to UTC epoch seconds for comparison against timestamp column
start_epoch_utc = int(start_sh.astimezone(pytz.UTC).timestamp())
end_epoch_utc = int(end_sh.astimezone(pytz.UTC).timestamp())

# Filter metric and trace within incident range (>= start, < end)
metric_in_range = metric_df[(metric_df['timestamp'] >= start_epoch_utc) & (metric_df['timestamp'] < end_epoch_utc)]
trace_in_range = trace_df[(trace_df['timestamp'] >= start_epoch_utc) & (trace_df['timestamp'] < end_epoch_utc)]

metric_in_count = int(metric_in_range.shape[0])
metric_in_cmdbs = metric_in_range['cmdb_id'].dropna().unique().tolist()
metric_in_top10 = (
    metric_in_range['kpi_name']
    .value_counts()
    .head(10)
    .reset_index()
    .rename(columns={'index':'kpi_name','kpi_name':'count'})
)
metric_in_top10_list = metric_in_top10.to_dict(orient='records')

trace_in_count = int(trace_in_range.shape[0])
trace_in_cmdbs = trace_in_range['cmdb_id'].dropna().unique().tolist()
trace_in_top10 = (
    trace_in_range['trace_name']
    .value_counts()
    .head(10)
    .reset_index()
    .rename(columns={'index':'trace_name','trace_name':'count'})
)
trace_in_top10_list = trace_in_top10.to_dict(orient='records')

# Assemble JSON-serializable compact result
result = {
    "metric_summary": {
        "total_rows": metric_total,
        "timestamp_min_asia_shanghai": metric_ts_min,
        "timestamp_max_asia_shanghai": metric_ts_max,
        "unique_cmdb_id": metric_unique_cmdb,
        "top_20_kpi_name_counts": metric_top_kpis_list
    },
    "trace_summary": {
        "total_rows": trace_total,
        "timestamp_min_asia_shanghai": trace_ts_min,
        "timestamp_max_asia_shanghai": trace_ts_max,
        "unique_cmdb_id": trace_unique_cmdb,
        "top_20_trace_name_counts": trace_top_names_list
    },
    "incident_range_utc8": {
        "range_start_iso_asia_shanghai": start_sh.isoformat(),
        "range_end_iso_asia_shanghai": end_sh.isoformat(),
        "metric_in_range": {
            "count": metric_in_count,
            "distinct_cmdb_id": metric_in_cmdbs,
            "top_10_kpi_name_counts_in_range": metric_in_top10_list
        },
        "trace_in_range": {
            "count": trace_in_count,
            "distinct_cmdb_id": trace_in_cmdbs,
            "top_10_trace_name_counts_in_range": trace_in_top10_list
        }
    }
}

result

```
Out[1]:
```
```python
# Using previous execution result stored in `result`
# Compose a plain-English concise summary string and display it.

metric_summary = result['metric_summary']
trace_summary = result['trace_summary']
incident = result['incident_range_utc8']

summary = (
    "Summary:\n"
    "- metric.csv: total rows = {m_total}. Timestamps range {m_min} to {m_max} (Asia/Shanghai). "
    "Unique cmdb_id count = {m_cmdb_count}. Top kpi counts (top values) are around: {m_top_samples}.\n"
    "- trace.csv: total rows = {t_total}. Timestamps range {t_min} to {t_max} (Asia/Shanghai). "
    "Unique cmdb_id count = {t_cmdb_count}. Top trace counts (top values) are around: {t_top_samples}.\n"
    "- Incident range (Asia/Shanghai {rng_start} to {rng_end}):\n"
    "  * metric rows in range = {m_in_count}, distinct cmdb_ids in-range = {m_in_cmdb_count}.\n"
    "    Top 10 kpi counts in-range (counts only) sample: {m_in_top_counts}.\n"
    "  * trace rows in range = {t_in_count}, distinct cmdb_ids in-range = {t_in_cmdb_count}.\n"
    "    Top 10 trace counts in-range (counts only) sample: {t_in_top_counts}.\n"
    "\nInterpretation notes: metric data covers the full hour (00:00–01:00), while traces are present from 00:30–00:59. "
    "There is a large volume of metric records in the incident window (~{m_in_count}) and a moderate number of trace records ({t_in_count}); "
    "top-count values indicate many KPI/trace series have similar per-series counts (repeated counts like 659 for KPIs and 224/208/180 for traces)."
).format(
    m_total=metric_summary['total_rows'],
    m_min=metric_summary['timestamp_min_asia_shanghai'],
    m_max=metric_summary['timestamp_max_asia_shanghai'],
    m_cmdb_count=len(metric_summary['unique_cmdb_id']),
    m_top_samples=[d.get('count') for d in metric_summary['top_20_kpi_name_counts'][:5]],
    t_total=trace_summary['total_rows'],
    t_min=trace_summary['timestamp_min_asia_shanghai'],
    t_max=trace_summary['timestamp_max_asia_shanghai'],
    t_cmdb_count=len(trace_summary['unique_cmdb_id']),
    t_top_samples=[d.get('count') for d in trace_summary['top_20_trace_name_counts'][:5]],
    rng_start=incident['range_start_iso_asia_shanghai'],
    rng_end=incident['range_end_iso_asia_shanghai'],
    m_in_count=incident['metric_in_range']['count'],
    m_in_cmdb_count=len(incident['metric_in_range']['distinct_cmdb_id']),
    m_in_top_counts=[d.get('count') for d in incident['metric_in_range']['top_10_kpi_name_counts_in_range']],
    t_in_count=incident['trace_in_range']['count'],
    t_in_cmdb_count=len(incident['trace_in_range']['distinct_cmdb_id']),
    t_in_top_counts=[d.get('count') for d in incident['trace_in_range']['top_10_trace_name_counts_in_range']]
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric_summary': {'total_rows': 90545, 'timestamp_min_asia_shanghai': '2020-05-23T00:00:00+08:00', 'timestamp_max_asia_shanghai': '2020-05-23T01:00:00+08:00', 'unique_cmdb_id': ['db_001', 'db_002', 'db_003', 'db_004', 'db_005', 'db_006', 'db_007', 'db_008', 'db_009', 'db_010', 'db_011', 'db_012', 'db_013', 'docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_001', 'os_002', 'os_003', 'os_004', 'os_005', 'os_006', 'os_007', 'os_008', 'os_009', 'os_010', 'os_011', 'os_012', 'os_013', 'os_014', 'os_015', 'os_016', 'os_017', 'os_018', 'os_019', 'os_020', 'os_021', 'os_022', 'osb_001', 'redis_001', 'redis_002', 'redis_003', 'redis_004', 'redis_005', 'redis_006', 'redis_007', 'redis_008', 'redis_009', 'redis_010', 'redis_011', 'redis_012'], 'top_20_kpi_name_counts': [{'count': 1321}, {'count': 1320}, {'count': 1319}, {'count': 1319}, {'count': 1319}, {'count': 1319}, {'count': 1318}, {'count': 1318}, {'count': 1318}, {'count': 1318}, {'count': 1318}, {'count': 1318}, {'count': 1318}, {'count': 1317}, {'count': 1316}, {'count': 1316}, {'count': 1316}, {'count': 1316}, {'count': 1316}, {'count': 1315}]}, 'trace_summary': {'total_rows': 6048, 'timestamp_min_asia_shanghai': '2020-05-23T00:30:00+08:00', 'timestamp_max_asia_shanghai': '2020-05-23T00:59:00+08:00', 'unique_cmdb_id': ['docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_021', 'os_022'], 'top_20_trace_name_counts': [{'count': 224}, {'count': 224}, {'count': 224}, {'count': 224}, {'count': 208}, {'count': 208}, {'count': 208}, {'count': 208}, {'count': 180}, {'count': 180}, {'count': 180}, {'count': 180}, {'count': 120}, {'count': 120}, {'count': 120}, {'count': 120}, {'count': 60}, {'count': 60}, {'count': 60}, {'count': 60}]}, 'incident_range_utc8': {'range_start_iso_asia_shanghai': '2020-05-23T00:30:00+08:00', 'range_end_iso_asia_shanghai': '2020-05-23T01:00:00+08:00', 'metric_in_range': {'count': 45358, 'distinct_cmdb_id': ['db_001', 'db_002', 'db_003', 'db_004', 'db_005', 'db_006', 'db_007', 'db_008', 'db_009', 'db_010', 'db_011', 'db_012', 'db_013', 'docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_001', 'os_002', 'os_003', 'os_004', 'os_005', 'os_006', 'os_007', 'os_008', 'os_009', 'os_010', 'os_011', 'os_012', 'os_013', 'os_014', 'os_015', 'os_016', 'os_017', 'os_018', 'os_019', 'os_020', 'os_021', 'os_022', 'osb_001', 'redis_001', 'redis_002', 'redis_003', 'redis_004', 'redis_005', 'redis_006', 'redis_007', 'redis_008', 'redis_009', 'redis_010', 'redis_011', 'redis_012'], 'top_10_kpi_name_counts_in_range': [{'count': 659}, {'count': 659}, {'count': 659}, {'count': 659}, {'count': 658}, {'count': 658}, {'count': 658}, {'count': 658}, {'count': 658}, {'count': 658}]}, 'trace_in_range': {'count': 6048, 'distinct_cmdb_id': ['docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_021', 'os_022'], 'top_10_trace_name_counts_in_range': [{'count': 224}, {'count': 224}, {'count': 224}, {'count': 224}, {'count': 208}, {'count': 208}, {'count': 208}, {'count': 208}, {'count': 180}, {'count': 180}]}}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime, timezone

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone and incident window (Asia/Shanghai UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_sh = tz_sh.localize(datetime(2020,5,23,0,30,0))
end_sh   = tz_sh.localize(datetime(2020,5,23,1,0,0))
start_epoch_utc = int(start_sh.astimezone(pytz.UTC).timestamp())
end_epoch_utc   = int(end_sh.astimezone(pytz.UTC).timestamp())

# 1) Compute global P95 and total points for each (cmdb_id, kpi_name)
grouped = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(total_points_full_series='count',
         global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter metric points to incident range (>= start, < end)
metric_in_range = metric_df[(metric_df['timestamp'] >= start_epoch_utc) & (metric_df['timestamp'] < end_epoch_utc)].copy()

# Merge global P95 into in-range points to evaluate breaches
metric_in_with_p95 = metric_in_range.merge(grouped, on=['cmdb_id','kpi_name'], how='left')

# Mark breaches: value > global_p95 (strictly exceed)
metric_in_with_p95['is_breach'] = metric_in_with_p95['value'] > metric_in_with_p95['global_p95']

# Aggregate in-range stats per group
in_range_agg = (
    metric_in_with_p95
    .groupby(['cmdb_id','kpi_name'], sort=False)
    .agg(
        in_range_point_count=('value','count'),
        in_range_breach_count=('is_breach','sum'),
        earliest_breach_ts=('timestamp', lambda s: s[metric_in_with_p95.loc[s.index, 'is_breach']].min() if (metric_in_with_p95.loc[s.index, 'is_breach'].any()) else pd.NA),
        in_range_max_value=('value','max')
    )
    .reset_index()
)

# Merge aggregated in-range stats back with global grouped info to include pairs with zero in-range points
merged = grouped.merge(in_range_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for groups with no in-range data
merged['in_range_point_count'] = merged['in_range_point_count'].fillna(0).astype(int)
merged['in_range_breach_count'] = merged['in_range_breach_count'].fillna(0).astype(int)
merged['in_range_max_value'] = merged['in_range_max_value'].where(merged['in_range_max_value'].notna(), None)

# Convert earliest breach timestamp to Asia/Shanghai ISO string or None
def ts_to_sh_iso(ts):
    if pd.isna(ts):
        return None
    try:
        dt_utc = datetime.fromtimestamp(int(ts), tz=timezone.utc)
        return dt_utc.astimezone(tz_sh).isoformat()
    except Exception:
        return None

merged['earliest_breach_timestamp_asia_shanghai'] = merged['earliest_breach_ts'].apply(ts_to_sh_iso)
merged.drop(columns=['earliest_breach_ts'], inplace=True)

# Reorder/select columns as required
final_cols = [
    'cmdb_id','kpi_name','total_points_full_series','global_p95',
    'in_range_point_count','in_range_breach_count','earliest_breach_timestamp_asia_shanghai','in_range_max_value'
]
final_df = merged[final_cols]

# 3) Top 20 pairs by in-range breach count (desc)
top20 = final_df.sort_values(by=['in_range_breach_count','in_range_point_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# 4) Small summary: total distinct component-KPI pairs processed
total_pairs = int(final_df.shape[0])

# Prepare JSON-serializable result: convert top20 to list of dicts with native Python types
top20_records = top20.to_dict(orient='records')

result_metric_p95 = {
    'top_20_by_in_range_breaches': top20_records,
    'total_distinct_component_kpi_pairs': total_pairs,
    'incident_range_asia_shanghai': {
        'start_iso': start_sh.isoformat(),
        'end_iso': end_sh.isoformat()
    }
}

result_metric_p95

```
Out[2]:
```
Incident window: 2020-05-23 00:30:00 to 2020-05-23 01:00:00 (Asia/Shanghai).

Key results
- Total distinct component–KPI pairs processed: 2,047.
- Top 20 pairs by in-range P95 breaches: every one of these top 20 had 3 breaches within the incident window.
- Common pattern for these top 20:
  - total_points_full_series = 60 (global series length),
  - in_range_point_count = 30 (points inside the 00:30–01:00 window),
  - in_range_breach_count = 3 (points > global P95).

Top 20 component–KPI pairs (cmdb_id — kpi_name : earliest breach timestamp — in-range max value)
1) db_001 — Exec_Per_Sec : 2020-05-23T00:38:00+08:00 — max 260.0  
2) db_001 — Logic_Read_Per_Sec : 2020-05-23T00:31:00+08:00 — max 9583.1  
3) db_001 — Physical_Read_Per_Sec : 2020-05-23T00:38:00+08:00 — max 229.7  
4) db_003 — Call_Per_Sec : 2020-05-23T00:52:00+08:00 — max 295.0  
5) db_003 — Exec_Per_Sec : 2020-05-23T00:39:00+08:00 — max 472.6  
6) db_003 — Login_Per_Sec : 2020-05-23T00:42:00+08:00 — max 13.7  
7) db_003 — Session_pct : 2020-05-23T00:35:00+08:00 — max 36.036  
8) db_003 — TPS_Per_Sec : 2020-05-23T00:52:00+08:00 — max 41.3  
9) db_004 — CPU_free_pct : 2020-05-23T00:55:00+08:00 — max 98.616996  
10) db_005 — Call_Per_Sec : 2020-05-23T00:36:00+08:00 — max 603.5  
11) db_005 — MEM_Used : 2020-05-23T00:52:00+08:00 — max 427.403715  
12) db_005 — MEM_Used_Pct : 2020-05-23T00:35:00+08:00 — max 84.322148  
13) db_005 — TPS_Per_Sec : 2020-05-23T00:36:00+08:00 — max 98.7  
14) db_006 — Call_Per_Sec : 2020-05-23T00:41:00+08:00 — max 547.0  
15) db_006 — TPS_Per_Sec : 2020-05-23T00:41:00+08:00 — max 89.9  
16) db_007 — CPU_free_pct : 2020-05-23T00:52:00+08:00 — max 98.613367  
17) db_007 — Call_Per_Sec : 2020-05-23T00:32:00+08:00 — max 79.4  
18) db_007 — MEM_Used_Pct : 2020-05-23T00:30:00+08:00 — max 85.019337  
19) db_007 — Physical_Read_Per_Sec : 2020-05-23T00:39:00+08:00 — max 351.1  
20) db_007 — TPS_Per_Sec : 2020-05-23T00:32:00+08:00 — max 11.9

Interpretation notes
- All top entries show the same breach count (3) over the 30-minute incident window, suggesting multiple KPI series exceeded their global P95 several times during the incident.
- Some KPIs show very large in-range maxima relative to their P95 (e.g., db_001 Logic_Read_Per_Sec max 9583.1), which may warrant deeper inspection of those specific metrics and timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

{'top_20_by_in_range_breaches': [{'cmdb_id': 'db_001', 'kpi_name': 'Exec_Per_Sec', 'total_points_full_series': 60, 'global_p95': 214.88499999999996, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00', 'in_range_max_value': 260.0}, {'cmdb_id': 'db_001', 'kpi_name': 'Logic_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95': 2401.3849999999998, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00', 'in_range_max_value': 9583.1}, {'cmdb_id': 'db_001', 'kpi_name': 'Physical_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95': 205.61999999999998, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00', 'in_range_max_value': 229.7}, {'cmdb_id': 'db_003', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95': 279.54, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00', 'in_range_max_value': 295.0}, {'cmdb_id': 'db_003', 'kpi_name': 'Exec_Per_Sec', 'total_points_full_series': 60, 'global_p95': 401.2799999999999, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:39:00+08:00', 'in_range_max_value': 472.6}, {'cmdb_id': 'db_003', 'kpi_name': 'Login_Per_Sec', 'total_points_full_series': 60, 'global_p95': 13.405, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:42:00+08:00', 'in_range_max_value': 13.7}, {'cmdb_id': 'db_003', 'kpi_name': 'Session_pct', 'total_points_full_series': 60, 'global_p95': 1.51405, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00', 'in_range_max_value': 36.036}, {'cmdb_id': 'db_003', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95': 39.13, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00', 'in_range_max_value': 41.3}, {'cmdb_id': 'db_004', 'kpi_name': 'CPU_free_pct', 'total_points_full_series': 60, 'global_p95': 98.3470856, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:55:00+08:00', 'in_range_max_value': 98.616996}, {'cmdb_id': 'db_005', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95': 577.13, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00', 'in_range_max_value': 603.5}, {'cmdb_id': 'db_005', 'kpi_name': 'MEM_Used', 'total_points_full_series': 60, 'global_p95': 427.35675454999995, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00', 'in_range_max_value': 427.403715}, {'cmdb_id': 'db_005', 'kpi_name': 'MEM_Used_Pct', 'total_points_full_series': 60, 'global_p95': 84.3154815, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00', 'in_range_max_value': 84.322148}, {'cmdb_id': 'db_005', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95': 94.32499999999999, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00', 'in_range_max_value': 98.7}, {'cmdb_id': 'db_006', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95': 542.025, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00', 'in_range_max_value': 547.0}, {'cmdb_id': 'db_006', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95': 89.005, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00', 'in_range_max_value': 89.9}, {'cmdb_id': 'db_007', 'kpi_name': 'CPU_free_pct', 'total_points_full_series': 60, 'global_p95': 98.51085675, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00', 'in_range_max_value': 98.613367}, {'cmdb_id': 'db_007', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95': 74.86, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00', 'in_range_max_value': 79.4}, {'cmdb_id': 'db_007', 'kpi_name': 'MEM_Used_Pct', 'total_points_full_series': 60, 'global_p95': 85.00024455, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'in_range_max_value': 85.019337}, {'cmdb_id': 'db_007', 'kpi_name': 'Physical_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95': 96.72999999999999, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:39:00+08:00', 'in_range_max_value': 351.1}, {'cmdb_id': 'db_007', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95': 11.014999999999999, 'in_range_point_count': 30, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00', 'in_range_max_value': 11.9}], 'total_distinct_component_kpi_pairs': 2047, 'incident_range_asia_shanghai': {'start_iso': '2020-05-23T00:30:00+08:00', 'end_iso': '2020-05-23T01:00:00+08:00'}}```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime, timezone

# Reuse trace_df if exists, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure correct dtypes
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Timezone and incident window (Asia/Shanghai UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_sh = tz_sh.localize(datetime(2020,5,23,0,30,0))
end_sh   = tz_sh.localize(datetime(2020,5,23,1,0,0))
start_epoch_utc = int(start_sh.astimezone(pytz.UTC).timestamp())
end_epoch_utc   = int(end_sh.astimezone(pytz.UTC).timestamp())

# 1) Compute global P95 and total points per (cmdb_id, trace_name)
trace_grouped = (
    trace_df
    .groupby(['cmdb_id','trace_name'], sort=False)['value']
    .agg(total_points_full_series='count',
         global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter trace points to incident window
trace_in_range = trace_df[(trace_df['timestamp'] >= start_epoch_utc) & (trace_df['timestamp'] < end_epoch_utc)].copy()

# Merge global P95 into in-range rows to evaluate breaches
trace_in_with_p95 = trace_in_range.merge(trace_grouped, on=['cmdb_id','trace_name'], how='left')

# Mark breaches: value > global_p95
trace_in_with_p95['is_breach'] = trace_in_with_p95['value'] > trace_in_with_p95['global_p95']

# Aggregate in-range stats per series
in_range_agg = (
    trace_in_with_p95
    .groupby(['cmdb_id','trace_name'], sort=False)
    .agg(
        in_range_point_count=('value','count'),
        in_range_breach_count=('is_breach','sum'),
        earliest_breach_ts=('timestamp', lambda s: s[trace_in_with_p95.loc[s.index, 'is_breach']].min() if (trace_in_with_p95.loc[s.index, 'is_breach'].any()) else pd.NA),
        in_range_max_value=('value','max')
    )
    .reset_index()
)

# Merge aggregated in-range stats back with global info to include series with zero in-range points
merged = trace_grouped.merge(in_range_agg, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for groups with no in-range data
merged['in_range_point_count'] = merged['in_range_point_count'].fillna(0).astype(int)
merged['in_range_breach_count'] = merged['in_range_breach_count'].fillna(0).astype(int)
merged['in_range_max_value'] = merged['in_range_max_value'].where(merged['in_range_max_value'].notna(), None)

# Convert earliest breach timestamp to Asia/Shanghai ISO string or None
def ts_to_sh_iso(ts):
    if pd.isna(ts):
        return None
    try:
        dt_utc = datetime.fromtimestamp(int(ts), tz=timezone.utc)
        return dt_utc.astimezone(tz_sh).isoformat()
    except Exception:
        return None

merged['earliest_breach_timestamp_asia_shanghai'] = merged['earliest_breach_ts'].apply(ts_to_sh_iso)
merged.drop(columns=['earliest_breach_ts'], inplace=True)

# Select and order columns for output
final_cols = [
    'cmdb_id','trace_name','total_points_full_series','global_p95',
    'in_range_point_count','in_range_breach_count','earliest_breach_timestamp_asia_shanghai','in_range_max_value'
]
final_df = merged[final_cols]

# 3) Top 20 series by in-range breach count (desc), tie-breaker by in_range_point_count desc
top20 = final_df.sort_values(by=['in_range_breach_count','in_range_point_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# 4) Summary stats
total_distinct_series = int(final_df.shape[0])
total_trace_rows_in_incident = int(trace_in_range.shape[0])

# Prepare JSON-serializable result
top20_records = top20.to_dict(orient='records')

result_trace_p95 = {
    'top_20_by_in_range_breaches': top20_records,
    'total_distinct_cmdb_trace_series': total_distinct_series,
    'total_trace_rows_in_incident_window': total_trace_rows_in_incident,
    'incident_range_asia_shanghai': {
        'start_iso': start_sh.isoformat(),
        'end_iso': end_sh.isoformat()
    }
}

result_trace_p95

```
Out[3]:
```
Incident window: 2020-05-23 00:30:00 to 2020-05-23 01:00:00 (Asia/Shanghai).

Key summary
- Total distinct cmdb_id–trace_name series processed: 208.
- Total trace rows within incident window: 6,048.
- Top 20 series by in-range P95 breaches: each shows 30 in-range points and 2 breaches (value > global P95).
- Most top entries belong to docker_001 and docker_002. Many of these traces are:
  - calls coming from os_022 (trace.from_os_022.remoteprocess.*),
  - self/internal traces (trace.self.csf.*, trace.self.flyremote.*),
  - outgoing calls to other containers (trace.to_docker_007.*, trace.to_docker_008.*),
  - and row_count series.

Selected highlights from the top 20 (cmdb_id — trace_name : global_P95 → in-range max → earliest breach)
- docker_001 — trace.from_os_022.remoteprocess.duration_mean : P95 ≈ 0.8553 → max 1.00497 → first breach 2020-05-23T00:37:00+08:00
- docker_001 — trace.from_os_022.remoteprocess.duration_p95 : P95 ≈ 2.6026 → max 2.73095 → first breach 2020-05-23T00:36:00+08:00
- docker_001 — trace.self.csf.duration_mean : P95 ≈ 0.07913 → max 0.083377 → first breach 2020-05-23T00:41:00+08:00
- docker_001 — trace.self.csf.row_count : P95 ≈ 591.4 → max 608.0 → first breach 2020-05-23T00:51:00+08:00
- docker_001 — trace.to_docker_007.remoteprocess.row_count : P95 ≈ 302.75 → max 315.0 → first breach 2020-05-23T00:51:00+08:00
- docker_002 — trace.from_os_022.remoteprocess.duration_p95 : P95 ≈ 2.5140 → max 3.07875 → first breach 2020-05-23T00:37:00+08:00
- docker_002 — trace.self.csf.duration_mean : P95 ≈ 0.09348 → max 0.110423 → first breach 2020-05-23T00:43:00+08:00

Interpretation notes
- The top trace series are fully present only in the incident window (30 total points each), and they had two P95 breaches each — a consistent pattern across affected docker nodes.
- Breaches include both latency-type measures (duration_mean / duration_p95) and row_count metrics, implying both timing and volume anomalies for inter-container calls involving docker_001/docker_002 and os_022 and for calls to docker_007/docker_008.
- These specific series (timestamps and maxima listed above) are good starting points for deeper investigation.

The original code execution output of IPython Kernel is also provided below for reference:

{'top_20_by_in_range_breaches': [{'cmdb_id': 'docker_001', 'trace_name': 'trace.from_os_022.remoteprocess.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.8553117999999995, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:37:00+08:00', 'in_range_max_value': 1.004971}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.from_os_022.remoteprocess.duration_p95', 'total_points_full_series': 30, 'global_p95': 2.6026124999999998, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00', 'in_range_max_value': 2.73095}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.self.csf.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.07913235, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00', 'in_range_max_value': 0.083377}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.self.csf.duration_p95', 'total_points_full_series': 30, 'global_p95': 0.1947525, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:45:00+08:00', 'in_range_max_value': 0.195}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.self.csf.row_count', 'total_points_full_series': 30, 'global_p95': 591.3999999999999, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:51:00+08:00', 'in_range_max_value': 608.0}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.self.flyremote.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.01478394999999999, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:40:00+08:00', 'in_range_max_value': 0.017082}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.self.flyremote.duration_p95', 'total_points_full_series': 30, 'global_p95': 0.091995, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:40:00+08:00', 'in_range_max_value': 0.097}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.to_docker_007.remoteprocess.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.068793, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00', 'in_range_max_value': 0.07}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.to_docker_007.remoteprocess.duration_p95', 'total_points_full_series': 30, 'global_p95': 0.07719999999999999, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00', 'in_range_max_value': 0.08}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.to_docker_007.remoteprocess.row_count', 'total_points_full_series': 30, 'global_p95': 302.75, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:51:00+08:00', 'in_range_max_value': 315.0}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.to_docker_008.remoteprocess.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.06819825, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00', 'in_range_max_value': 0.069854}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.to_docker_008.remoteprocess.duration_p95', 'total_points_full_series': 30, 'global_p95': 0.07764499999999999, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00', 'in_range_max_value': 0.08}, {'cmdb_id': 'docker_001', 'trace_name': 'trace.to_docker_008.remoteprocess.row_count', 'total_points_full_series': 30, 'global_p95': 287.65, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:51:00+08:00', 'in_range_max_value': 303.0}, {'cmdb_id': 'docker_002', 'trace_name': 'trace.from_os_022.remoteprocess.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.7900566499999998, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:37:00+08:00', 'in_range_max_value': 0.854143}, {'cmdb_id': 'docker_002', 'trace_name': 'trace.from_os_022.remoteprocess.duration_p95', 'total_points_full_series': 30, 'global_p95': 2.5140049999999996, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:37:00+08:00', 'in_range_max_value': 3.07875}, {'cmdb_id': 'docker_002', 'trace_name': 'trace.from_os_022.remoteprocess.row_count', 'total_points_full_series': 30, 'global_p95': 149.64999999999998, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:51:00+08:00', 'in_range_max_value': 162.0}, {'cmdb_id': 'docker_002', 'trace_name': 'trace.self.csf.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.09348044999999999, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:43:00+08:00', 'in_range_max_value': 0.110423}, {'cmdb_id': 'docker_002', 'trace_name': 'trace.self.csf.duration_p95', 'total_points_full_series': 30, 'global_p95': 0.19655, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:40:00+08:00', 'in_range_max_value': 0.2026}, {'cmdb_id': 'docker_002', 'trace_name': 'trace.self.csf.row_count', 'total_points_full_series': 30, 'global_p95': 598.5999999999999, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:51:00+08:00', 'in_range_max_value': 648.0}, {'cmdb_id': 'docker_002', 'trace_name': 'trace.self.flyremote.duration_mean', 'total_points_full_series': 30, 'global_p95': 0.016903449999999993, 'in_range_point_count': 30, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:44:00+08:00', 'in_range_max_value': 0.018285}], 'total_distinct_cmdb_trace_series': 208, 'total_trace_rows_in_incident_window': 6048, 'incident_range_asia_shanghai': {'start_iso': '2020-05-23T00:30:00+08:00', 'end_iso': '2020-05-23T01:00:00+08:00'}}```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse final_df from previous steps if available; otherwise reconstruct minimal needed info
try:
    final_df
except NameError:
    # fallback: recompute from metric_df and grouped/merged if available
    # This fallback assumes metric_df and grouped exist from previous runs.
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    tz_sh = pytz.timezone('Asia/Shanghai')
    start_sh = tz_sh.localize(datetime(2020,5,23,0,30,0))
    end_sh   = tz_sh.localize(datetime(2020,5,23,1,0,0))
    start_epoch_utc = int(start_sh.astimezone(pytz.UTC).timestamp())
    end_epoch_utc   = int(end_sh.astimezone(pytz.UTC).timestamp())
    grouped = (
        metric_df
        .groupby(['cmdb_id','kpi_name'], sort=False)['value']
        .agg(total_points_full_series='count',
             global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )
    metric_in_range = metric_df[(metric_df['timestamp'] >= start_epoch_utc) & (metric_df['timestamp'] < end_epoch_utc)].copy()
    metric_in_with_p95 = metric_in_range.merge(grouped, on=['cmdb_id','kpi_name'], how='left')
    metric_in_with_p95['is_breach'] = metric_in_with_p95['value'] > metric_in_with_p95['global_p95']
    in_range_agg = (
        metric_in_with_p95
        .groupby(['cmdb_id','kpi_name'], sort=False)
        .agg(
            in_range_point_count=('value','count'),
            in_range_breach_count=('is_breach','sum'),
            earliest_breach_ts=('timestamp', lambda s: s[metric_in_with_p95.loc[s.index, 'is_breach']].min() if (metric_in_with_p95.loc[s.index, 'is_breach'].any()) else pd.NA),
            in_range_max_value=('value','max')
        )
        .reset_index()
    )
    merged = grouped.merge(in_range_agg, on=['cmdb_id','kpi_name'], how='left')
    merged['in_range_point_count'] = merged['in_range_point_count'].fillna(0).astype(int)
    merged['in_range_breach_count'] = merged['in_range_breach_count'].fillna(0).astype(int)
    merged['in_range_max_value'] = merged['in_range_max_value'].where(merged['in_range_max_value'].notna(), None)
    def ts_to_sh_iso(ts):
        if pd.isna(ts):
            return None
        try:
            return datetime.utcfromtimestamp(int(ts)).replace(tzinfo=pytz.UTC).astimezone(tz_sh).isoformat()
        except Exception:
            return None
    merged['earliest_breach_timestamp_asia_shanghai'] = merged['earliest_breach_ts'].apply(ts_to_sh_iso)
    merged.drop(columns=['earliest_breach_ts'], inplace=True)
    final_cols = [
        'cmdb_id','kpi_name','total_points_full_series','global_p95',
        'in_range_point_count','in_range_breach_count','earliest_breach_timestamp_asia_shanghai','in_range_max_value'
    ]
    final_df = merged[final_cols]

# Candidate DBs
db_candidates = ['db_001','db_003','db_005','db_006','db_007']

# Filter to candidates and to series that had in-range breaches (>0)
df_db = final_df[
    final_df['cmdb_id'].isin(db_candidates) &
    (final_df['in_range_breach_count'] > 0)
].copy()

# Compute breach_severity_ratio = (in_range_max_value - global_p95)/global_p95
# Handle possible None or zero global_p95
def compute_ratio(row):
    gp = row['global_p95']
    mv = row['in_range_max_value']
    if gp is None or mv is None:
        return None
    try:
        gp_f = float(gp)
        if gp_f == 0:
            return None
        return float((mv - gp_f) / gp_f)
    except Exception:
        return None

df_db['breach_severity_ratio'] = df_db.apply(compute_ratio, axis=1)

# Sort by in_range_breach_count desc then breach_severity_ratio desc
df_db = df_db.sort_values(by=['in_range_breach_count','breach_severity_ratio'], ascending=[False, False]).head(100)

# Prepare compact JSON-serializable rows (cast to native types)
rows = []
for _, r in df_db.iterrows():
    rows.append({
        'cmdb_id': str(r['cmdb_id']),
        'kpi_name': str(r['kpi_name']),
        'global_p95_value': None if pd.isna(r['global_p95']) else float(r['global_p95']),
        'in_range_max_value': None if r['in_range_max_value'] is None or pd.isna(r['in_range_max_value']) else float(r['in_range_max_value']),
        'earliest_breach_timestamp_asia_shanghai': r['earliest_breach_timestamp_asia_shanghai'] if pd.notna(r['earliest_breach_timestamp_asia_shanghai']) else None,
        'in_range_breach_count': int(r['in_range_breach_count']),
        'total_points_full_series': int(r['total_points_full_series']),
        'breach_severity_ratio': None if r['breach_severity_ratio'] is None or (isinstance(r['breach_severity_ratio'], float) and (np.isnan(r['breach_severity_ratio']))) else float(r['breach_severity_ratio'])
    })

# Summary calculations
# earliest breach timestamp among these DB candidates
if len(rows) > 0:
    timestamps = [pd.to_datetime(r['earliest_breach_timestamp_asia_shanghai']) for r in rows if r['earliest_breach_timestamp_asia_shanghai'] is not None]
    earliest_breach_ts = None
    if timestamps:
        earliest_breach_ts = min(timestamps).tz_convert(pytz.timezone('Asia/Shanghai')).isoformat()
    # DB candidate with largest breach_severity_ratio
    ratios = [(r['cmdb_id'], r['breach_severity_ratio']) for r in rows if r['breach_severity_ratio'] is not None]
    if ratios:
        db_max, max_ratio = max(ratios, key=lambda x: x[1])
    else:
        db_max, max_ratio = None, None
else:
    earliest_breach_ts = None
    db_max, max_ratio = None, None

summary = {
    'earliest_breach_timestamp_among_candidates': earliest_breach_ts,
    'db_with_largest_breach_severity_ratio': db_max,
    'largest_breach_severity_ratio': None if max_ratio is None else float(max_ratio),
    'distinct_kpi_series_returned': len(rows)
}

result_db_kpis = {
    'rows': rows,
    'summary': summary,
    'incident_range_asia_shanghai': {'start_iso': '2020-05-23T00:30:00+08:00', 'end_iso': '2020-05-23T01:00:00+08:00'}
}

result_db_kpis

```
Out[4]:
```
Summary (straightforward):

- Incident window analyzed: 2020-05-23 00:30:00 to 2020-05-23 01:00:00 (Asia/Shanghai).
- DB candidates checked: db_001, db_003, db_005, db_006, db_007.
- Result: No KPI series for these DB candidates exceeded their global P95 during the incident window.
  - Rows returned: 0 (no in-range P95 breaches found).
  - Earliest breach among these candidates: none.
  - DB with largest breach severity ratio: none (no breaches).
- Conclusion: For the requested DB candidate set, there are no in-window P95 breaches to investigate further. If you want, I can (a) list all KPIs for these DBs and their in-range stats, or (b) lower the threshold (e.g., P90) to find smaller anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

{'rows': [], 'summary': {'earliest_breach_timestamp_among_candidates': None, 'db_with_largest_breach_severity_ratio': None, 'largest_breach_severity_ratio': None, 'distinct_kpi_series_returned': 0}, 'incident_range_asia_shanghai': {'start_iso': '2020-05-23T00:30:00+08:00', 'end_iso': '2020-05-23T01:00:00+08:00'}}```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np
from datetime import datetime, timezone

# Reuse metric_df from previous state if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure correct dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# DB candidates
db_candidates = ['db_001','db_003','db_005','db_006','db_007']

# Timezone and incident window (Asia/Shanghai UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_sh = tz_sh.localize(datetime(2020,5,23,0,30,0))
end_sh   = tz_sh.localize(datetime(2020,5,23,1,0,0))
start_epoch_utc = int(start_sh.astimezone(pytz.UTC).timestamp())
end_epoch_utc   = int(end_sh.astimezone(pytz.UTC).timestamp())

# 1) Compute global P95 and total points for each (cmdb_id, kpi_name) restricted to DB candidates
grouped_global = (
    metric_df[metric_df['cmdb_id'].isin(db_candidates)]
    .groupby(['cmdb_id','kpi_name'], sort=False)['value']
    .agg(total_points_full_series='count',
         global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter metric points to incident range for these DBs and merge global thresholds
metric_in_range = metric_df[
    (metric_df['cmdb_id'].isin(db_candidates)) &
    (metric_df['timestamp'] >= start_epoch_utc) &
    (metric_df['timestamp'] < end_epoch_utc)
].copy()

# Merge global p95 onto in-range rows to evaluate breaches
metric_in_with_p95 = metric_in_range.merge(grouped_global, on=['cmdb_id','kpi_name'], how='left')

# Mark breaches: value > global_p95
metric_in_with_p95['is_breach'] = metric_in_with_p95['value'] > metric_in_with_p95['global_p95']

# Aggregate in-range stats per (cmdb_id,kpi_name)
in_range_agg = (
    metric_in_with_p95
    .groupby(['cmdb_id','kpi_name'], sort=False)
    .agg(
        in_range_point_count=('value','count'),
        in_range_max_value=('value','max'),
        in_range_breach_count=('is_breach','sum'),
        earliest_breach_ts=('timestamp', lambda s: s[metric_in_with_p95.loc[s.index,'is_breach']].min() if (metric_in_with_p95.loc[s.index,'is_breach'].any()) else pd.NA)
    )
    .reset_index()
)

# Merge aggregated in-range stats with global (to include KPI series with zero in-range points)
merged = grouped_global.merge(in_range_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for groups with no in-range data
merged['in_range_point_count'] = merged['in_range_point_count'].fillna(0).astype(int)
merged['in_range_breach_count'] = merged['in_range_breach_count'].fillna(0).astype(int)
merged['in_range_max_value'] = merged['in_range_max_value'].where(merged['in_range_max_value'].notna(), None)

# Convert earliest breach timestamp to Asia/Shanghai ISO string or None
def ts_to_sh_iso(ts):
    if pd.isna(ts):
        return None
    try:
        dt_utc = datetime.fromtimestamp(int(ts), tz=timezone.utc)
        return dt_utc.astimezone(tz_sh).isoformat()
    except Exception:
        return None

merged['earliest_breach_timestamp_asia_shanghai'] = merged['earliest_breach_ts'].apply(ts_to_sh_iso)
merged.drop(columns=['earliest_breach_ts'], inplace=True)

# Reorder/select required columns
output_cols = [
    'cmdb_id','kpi_name','total_points_full_series','global_p95',
    'in_range_point_count','in_range_max_value','in_range_breach_count','earliest_breach_timestamp_asia_shanghai'
]
out_df = merged[output_cols].copy()

# Compute breach severity ratio for possible sorting (not required to return but useful)
def severity_ratio(row):
    gp = row['global_p95']
    mv = row['in_range_max_value']
    if pd.isna(gp) or gp == 0 or mv is None:
        return np.nan
    return (float(mv) - float(gp)) / float(gp)

out_df['breach_severity_ratio'] = out_df.apply(severity_ratio, axis=1)

# Sort by in_range_breach_count desc then severity desc, limit to 100 rows
out_df = out_df.sort_values(by=['in_range_breach_count','breach_severity_ratio'], ascending=[False, False]).head(100).reset_index(drop=True)

# Prepare JSON-serializable rows
rows = []
for _, r in out_df.iterrows():
    rows.append({
        'cmdb_id': str(r['cmdb_id']),
        'kpi_name': str(r['kpi_name']),
        'total_points_full_series': int(r['total_points_full_series']),
        'global_p95_value': None if pd.isna(r['global_p95']) else float(r['global_p95']),
        'in_range_point_count': int(r['in_range_point_count']),
        'in_range_max_value': None if r['in_range_max_value'] is None or pd.isna(r['in_range_max_value']) else float(r['in_range_max_value']),
        'in_range_breach_count': int(r['in_range_breach_count']),
        'earliest_breach_timestamp_asia_shanghai': r['earliest_breach_timestamp_asia_shanghai'] if pd.notna(r['earliest_breach_timestamp_asia_shanghai']) else None
    })

# Summary
number_of_kpi_series_returned = len(rows)
total_in_range_breaches_across_all_returned_series = int(out_df['in_range_breach_count'].sum())

result_db_kpi_summary = {
    'rows': rows,
    'summary': {
        'number_of_kpi_series_returned': number_of_kpi_series_returned,
        'total_in_range_breaches_across_all_returned_series': total_in_range_breaches_across_all_returned_series
    },
    'incident_range_asia_shanghai': {'start_iso': start_sh.isoformat(), 'end_iso': end_sh.isoformat()}
}

result_db_kpi_summary

```
Out[5]:
```
Incident window: 2020-05-23 00:30:00 – 01:00:00 (Asia/Shanghai)

Key numeric results
- Rows returned (one row per DB kpi_name, limited to 100): 100.
- Total distinct KPI series returned: 100.
- Total in-range P95 breaches summed across these returned series: 156.
- Earliest breach timestamp among these DB candidates: 2020-05-23T00:30:00+08:00.

Notable KPI anomalies (selected high-impact examples from the returned rows)
- db_001 — Logic_Read_Per_Sec: global P95 ≈ 2401.38 → in-range max 9583.1; earliest breach 2020-05-23T00:31:00+08:00; breaches = 3.
- db_007 — Logic_Read_Per_Sec: global P95 ≈ 3882.75 → in-range max 36,564.7; earliest breach 2020-05-23T00:38:00+08:00; breaches = 2.
- db_003 — Physical_Read_Per_Sec: global P95 ≈ 2414.82 → in-range max 5726.1; earliest breach 2020-05-23T00:32:00+08:00; breaches = 2.
- db_007 — Physical_Read_Per_Sec: global P95 ≈ 96.73 → in-range max 351.1; earliest breach 2020-05-23T00:39:00+08:00; breaches = 3.
- db_003 — Session_pct: global P95 ≈ 1.514 → in-range max 36.036; earliest breach 2020-05-23T00:35:00+08:00; breaches = 3.
- db_003 — DbTime: global P95 ≈ 298.69 → in-range max 2991.3; earliest breach 2020-05-23T00:35:00+08:00; breaches = 2.
- db_001 — DFParaWrite_Per_Sec: global P95 ≈ 1.75 → in-range max 42.25; earliest breach 2020-05-23T00:30:00+08:00; breaches = 2.
- db_003 / db_001 / db_005 / db_006 / db_007: multiple other KPIs (Redo_Per_Sec, Exec_Per_Sec, Call_Per_Sec, TPS_Per_Sec, MEM_Used*, CPU_free_pct, etc.) show 1–3 breaches and notable max values.

Interpretation / next steps
- Many DB KPIs across the candidate set had multiple P95 breaches during the 30-minute window; several breaches are large in magnitude (in-range max substantially above global P95), especially for read/redo/exec metrics and some session/row-count KPIs.
- Earliest symptoms appear right at the start of the window (00:30), so investigate timestamps around 00:30–00:40 first.
- Recommended follow-ups: inspect time-aligned raw series and traces for the highlighted KPIs (starting with Logic_Read_Per_Sec, Physical_Read_Per_Sec, Session_pct, DbTime, Redo_Per_Sec) on db_001, db_003, db_007 to determine cause (query storms, backups, I/O pressure, or other upstream events).

The original code execution output of IPython Kernel is also provided below for reference:

{'rows': [{'cmdb_id': 'db_003', 'kpi_name': 'Session_pct', 'total_points_full_series': 60, 'global_p95_value': 1.51405, 'in_range_point_count': 30, 'in_range_max_value': 36.036, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'Logic_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 2401.3849999999998, 'in_range_point_count': 30, 'in_range_max_value': 9583.1, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Physical_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 96.72999999999999, 'in_range_point_count': 30, 'in_range_max_value': 351.1, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:39:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'Exec_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 214.88499999999996, 'in_range_point_count': 30, 'in_range_max_value': 260.0, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Exec_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 401.2799999999999, 'in_range_point_count': 30, 'in_range_max_value': 472.6, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:39:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'Physical_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 205.61999999999998, 'in_range_point_count': 30, 'in_range_max_value': 229.7, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 11.014999999999999, 'in_range_point_count': 30, 'in_range_max_value': 11.9, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 74.86, 'in_range_point_count': 30, 'in_range_max_value': 79.4, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 39.13, 'in_range_point_count': 30, 'in_range_max_value': 41.3, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 279.54, 'in_range_point_count': 30, 'in_range_max_value': 295.0, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 94.32499999999999, 'in_range_point_count': 30, 'in_range_max_value': 98.7, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 577.13, 'in_range_point_count': 30, 'in_range_max_value': 603.5, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Login_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 13.405, 'in_range_point_count': 30, 'in_range_max_value': 13.7, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:42:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'TPS_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 89.005, 'in_range_point_count': 30, 'in_range_max_value': 89.9, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 542.025, 'in_range_point_count': 30, 'in_range_max_value': 547.0, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:41:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'CPU_free_pct', 'total_points_full_series': 60, 'global_p95_value': 98.51085675, 'in_range_point_count': 30, 'in_range_max_value': 98.613367, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'MEM_Used_Pct', 'total_points_full_series': 60, 'global_p95_value': 85.00024455, 'in_range_point_count': 30, 'in_range_max_value': 85.019337, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'MEM_Used', 'total_points_full_series': 60, 'global_p95_value': 427.35675454999995, 'in_range_point_count': 30, 'in_range_max_value': 427.403715, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:52:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'MEM_Used_Pct', 'total_points_full_series': 60, 'global_p95_value': 84.3154815, 'in_range_point_count': 30, 'in_range_max_value': 84.322148, 'in_range_breach_count': 3, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'DFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 1.7505, 'in_range_point_count': 30, 'in_range_max_value': 42.25, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'DbTime', 'total_points_full_series': 60, 'global_p95_value': 298.69499999999977, 'in_range_point_count': 30, 'in_range_max_value': 2991.3, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Sess_Active', 'total_points_full_series': 60, 'global_p95_value': 3.034999999999998, 'in_range_point_count': 30, 'in_range_max_value': 29.9, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Logic_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 3882.7499999999995, 'in_range_point_count': 30, 'in_range_max_value': 36564.7, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Physical_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 2414.819999999999, 'in_range_point_count': 30, 'in_range_max_value': 5726.1, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'LFSync_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 29.589499999999994, 'in_range_point_count': 30, 'in_range_max_value': 60.13, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'LFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 10.7615, 'in_range_point_count': 30, 'in_range_max_value': 20.81, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'DFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 24.965999999999983, 'in_range_point_count': 30, 'in_range_max_value': 41.89, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:33:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'SeqRead_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 2.05, 'in_range_point_count': 30, 'in_range_max_value': 3.04, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:34:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Redo_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 24327.64999999998, 'in_range_point_count': 30, 'in_range_max_value': 34619.8, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'Redo_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 19752.215, 'in_range_point_count': 30, 'in_range_max_value': 25413.1, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Exec_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 128.79, 'in_range_point_count': 30, 'in_range_max_value': 151.3, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Redo_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 28686.0, 'in_range_point_count': 30, 'in_range_max_value': 33108.4, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'DFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 3.6124999999999994, 'in_range_point_count': 30, 'in_range_max_value': 3.99, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:44:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'LFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 1.793, 'in_range_point_count': 30, 'in_range_max_value': 1.87, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'CPU_free_pct', 'total_points_full_series': 61, 'global_p95_value': 98.943751, 'in_range_point_count': 30, 'in_range_max_value': 99.303632, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:50:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Sess_Connect', 'total_points_full_series': 60, 'global_p95_value': 412.0, 'in_range_point_count': 30, 'in_range_max_value': 413.0, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'CPU_Used_Pct', 'total_points_full_series': 43, 'global_p95_value': 4.2491183999999995, 'in_range_point_count': 23, 'in_range_max_value': 4.259376, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'CPU_free_pct', 'total_points_full_series': 61, 'global_p95_value': 98.962801, 'in_range_point_count': 30, 'in_range_max_value': 99.024211, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'CPU_free_pct', 'total_points_full_series': 60, 'global_p95_value': 97.95070795000001, 'in_range_point_count': 30, 'in_range_max_value': 98.00232, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:33:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'CPU_free_pct', 'total_points_full_series': 60, 'global_p95_value': 98.33427809999999, 'in_range_point_count': 30, 'in_range_max_value': 98.363269, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'MEM_Used', 'total_points_full_series': 60, 'global_p95_value': 373.9040633, 'in_range_point_count': 30, 'in_range_max_value': 373.979625, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'MEM_Used', 'total_points_full_series': 60, 'global_p95_value': 430.9213479, 'in_range_point_count': 30, 'in_range_max_value': 431.006177, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'MEM_Used', 'total_points_full_series': 60, 'global_p95_value': 177.1786147, 'in_range_point_count': 30, 'in_range_max_value': 177.181276, 'in_range_breach_count': 2, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:55:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'Login_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 0.3, 'in_range_point_count': 30, 'in_range_max_value': 1.6, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'LFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 27.07849999999994, 'in_range_point_count': 30, 'in_range_max_value': 63.91, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'AIOS', 'total_points_full_series': 60, 'global_p95_value': 0.03, 'in_range_point_count': 30, 'in_range_max_value': 0.06, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Login_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 0.10499999999999972, 'in_range_point_count': 30, 'in_range_max_value': 0.2, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Sess_Active', 'total_points_full_series': 60, 'global_p95_value': 0.7049999999999996, 'in_range_point_count': 30, 'in_range_max_value': 1.3, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Sess_Used_Temp', 'total_points_full_series': 30, 'global_p95_value': 0.5499999999999972, 'in_range_point_count': 15, 'in_range_max_value': 1.0, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'SeqRead_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 7.0954999999999995, 'in_range_point_count': 30, 'in_range_max_value': 12.78, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:37:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'DbTime', 'total_points_full_series': 60, 'global_p95_value': 72.92499999999998, 'in_range_point_count': 30, 'in_range_max_value': 126.3, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Session_pct', 'total_points_full_series': 60, 'global_p95_value': 0.17874999999999994, 'in_range_point_count': 30, 'in_range_max_value': 0.307, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'Call_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 11.305, 'in_range_point_count': 30, 'in_range_max_value': 17.7, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'ACS', 'total_points_full_series': 60, 'global_p95_value': 0.02, 'in_range_point_count': 30, 'in_range_max_value': 0.03, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:44:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Sess_Active', 'total_points_full_series': 60, 'global_p95_value': 0.2, 'in_range_point_count': 30, 'in_range_max_value': 0.3, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'LFSync_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 9.109499999999992, 'in_range_point_count': 30, 'in_range_max_value': 13.58, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Sess_Active', 'total_points_full_series': 60, 'global_p95_value': 0.20499999999999974, 'in_range_point_count': 30, 'in_range_max_value': 0.3, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'CPU_Used_Pct', 'total_points_full_series': 44, 'global_p95_value': 2.1469137, 'in_range_point_count': 24, 'in_range_max_value': 3.1271, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Redo_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 17809.159999999985, 'in_range_point_count': 30, 'in_range_max_value': 25108.9, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'ACS', 'total_points_full_series': 60, 'global_p95_value': 0.03, 'in_range_point_count': 30, 'in_range_max_value': 0.04, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:59:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'LFSync_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 2.3324999999999987, 'in_range_point_count': 30, 'in_range_max_value': 3.08, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:46:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'User_Commit', 'total_points_full_series': 60, 'global_p95_value': 0.25, 'in_range_point_count': 30, 'in_range_max_value': 0.33, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:47:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'Session_pct', 'total_points_full_series': 60, 'global_p95_value': 0.05674999999999996, 'in_range_point_count': 30, 'in_range_max_value': 0.074, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'DbTime', 'total_points_full_series': 60, 'global_p95_value': 22.914999999999985, 'in_range_point_count': 30, 'in_range_max_value': 29.8, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'DbTime', 'total_points_full_series': 60, 'global_p95_value': 23.705, 'in_range_point_count': 30, 'in_range_max_value': 28.5, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Session_pct', 'total_points_full_series': 60, 'global_p95_value': 0.058, 'in_range_point_count': 30, 'in_range_max_value': 0.067, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Redo_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 22902.78, 'in_range_point_count': 30, 'in_range_max_value': 25752.8, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Physical_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 178.95999999999998, 'in_range_point_count': 30, 'in_range_max_value': 196.7, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'CPU_Used_Pct', 'total_points_full_series': 43, 'global_p95_value': 2.0716047, 'in_range_point_count': 23, 'in_range_max_value': 2.241337, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:47:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Physical_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 65.88499999999999, 'in_range_point_count': 30, 'in_range_max_value': 71.2, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'SeqRead_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 1.7674999999999996, 'in_range_point_count': 30, 'in_range_max_value': 1.91, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Exec_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 362.54, 'in_range_point_count': 30, 'in_range_max_value': 385.7, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'LFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 4.784499999999999, 'in_range_point_count': 30, 'in_range_max_value': 5.06, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Logic_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 1761.1799999999996, 'in_range_point_count': 30, 'in_range_max_value': 1859.6, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'User_Commit', 'total_points_full_series': 60, 'global_p95_value': 1.18, 'in_range_point_count': 30, 'in_range_max_value': 1.23, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Exec_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 250.4, 'in_range_point_count': 30, 'in_range_max_value': 261.0, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'User_Commit', 'total_points_full_series': 60, 'global_p95_value': 1.23, 'in_range_point_count': 30, 'in_range_max_value': 1.27, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Logic_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 3075.2099999999996, 'in_range_point_count': 30, 'in_range_max_value': 3166.6, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'LFSync_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 2.5835, 'in_range_point_count': 30, 'in_range_max_value': 2.65, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:32:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'Logic_Read_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 108551.90499999998, 'in_range_point_count': 30, 'in_range_max_value': 110775.0, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'CPU_Used_Pct', 'total_points_full_series': 42, 'global_p95_value': 2.7694664, 'in_range_point_count': 23, 'in_range_max_value': 2.820142, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'DFParaWrite_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 1.44, 'in_range_point_count': 30, 'in_range_max_value': 1.46, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'SctRead_Per_Sec', 'total_points_full_series': 60, 'global_p95_value': 8.0525, 'in_range_point_count': 30, 'in_range_max_value': 8.1, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'UndoTbs_Pct', 'total_points_full_series': 2, 'global_p95_value': 0.309, 'in_range_point_count': 1, 'in_range_max_value': 0.31, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:57:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'MEM_real_util', 'total_points_full_series': 60, 'global_p95_value': 46.1905, 'in_range_point_count': 30, 'in_range_max_value': 46.31, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'Sess_Connect', 'total_points_full_series': 60, 'global_p95_value': 421.0, 'in_range_point_count': 30, 'in_range_max_value': 422.0, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:57:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'MEM_real_util', 'total_points_full_series': 60, 'global_p95_value': 46.722, 'in_range_point_count': 30, 'in_range_max_value': 46.81, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:48:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'MEM_Used', 'total_points_full_series': 60, 'global_p95_value': 245.36953575, 'in_range_point_count': 30, 'in_range_max_value': 245.778357, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:49:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'MEM_Used_Pct', 'total_points_full_series': 59, 'global_p95_value': 97.80993219999999, 'in_range_point_count': 29, 'in_range_max_value': 97.930815, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:39:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'UndoTbs_Pct', 'total_points_full_series': 2, 'global_p95_value': 2.4290000000000003, 'in_range_point_count': 1, 'in_range_max_value': 2.43, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:57:00+08:00'}, {'cmdb_id': 'db_005', 'kpi_name': 'Tbs_Free_Gb', 'total_points_full_series': 2, 'global_p95_value': 1.2995, 'in_range_point_count': 1, 'in_range_max_value': 1.3, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:57:00+08:00'}, {'cmdb_id': 'db_007', 'kpi_name': 'MEM_real_util', 'total_points_full_series': 61, 'global_p95_value': 82.38, 'in_range_point_count': 30, 'in_range_max_value': 82.4, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_006', 'kpi_name': 'MEM_Used_Pct', 'total_points_full_series': 60, 'global_p95_value': 34.863948, 'in_range_point_count': 30, 'in_range_max_value': 34.864492, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:39:00+08:00'}, {'cmdb_id': 'db_003', 'kpi_name': 'New_Tbs_Free_Gb', 'total_points_full_series': 6, 'global_p95_value': 195.8607031, 'in_range_point_count': 3, 'in_range_max_value': 195.8609531, 'in_range_breach_count': 1, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:57:00+08:00'}, {'cmdb_id': 'db_001', 'kpi_name': 'AIOS', 'total_points_full_series': 60, 'global_p95_value': 0.01, 'in_range_point_count': 30, 'in_range_max_value': 0.01, 'in_range_breach_count': 0, 'earliest_breach_timestamp_asia_shanghai': None}, {'cmdb_id': 'db_001', 'kpi_name': 'AWS', 'total_points_full_series': 60, 'global_p95_value': 0.02, 'in_range_point_count': 30, 'in_range_max_value': 0.02, 'in_range_breach_count': 0, 'earliest_breach_timestamp_asia_shanghai': None}, {'cmdb_id': 'db_001', 'kpi_name': 'Hang', 'total_points_full_series': 60, 'global_p95_value': 1.0, 'in_range_point_count': 30, 'in_range_max_value': 1.0, 'in_range_breach_count': 0, 'earliest_breach_timestamp_asia_shanghai': None}, {'cmdb_id': 'db_001', 'kpi_name': 'MEM_Total', 'total_points_full_series': 59, 'global_p95_value': 381.902264, 'in_range_point_count': 29, 'in_range_max_value': 381.902264, 'in_range_breach_count': 0, 'earliest_breach_timestamp_asia_shanghai': None}, {'cmdb_id': 'db_001', 'kpi_name': 'On_Off_State', 'total_points_full_series': 60, 'global_p95_value': 1.0, 'in_range_point_count': 30, 'in_range_max_value': 1.0, 'in_range_breach_count': 0, 'earliest_breach_timestamp_asia_shanghai': None}, {'cmdb_id': 'db_001', 'kpi_name': 'Sess_Active', 'total_points_full_series': 60, 'global_p95_value': 0.5, 'in_range_point_count': 30, 'in_range_max_value': 0.5, 'in_range_breach_count': 0, 'earliest_breach_timestamp_asia_shanghai': None}], 'summary': {'number_of_kpi_series_returned': 100, 'total_in_range_breaches_across_all_returned_series': 156}, 'incident_range_asia_shanghai': {'start_iso': '2020-05-23T00:30:00+08:00', 'end_iso': '2020-05-23T01:00:00+08:00'}}```
```

In [ ]:
# In[6]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[6]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from datetime import datetime, timezone

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone and incident window (Asia/Shanghai UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_sh = tz_sh.localize(datetime(2020,5,23,0,30,0))
end_sh   = tz_sh.localize(datetime(2020,5,23,1,0,0))
start_epoch_utc = int(start_sh.astimezone(pytz.UTC).timestamp())
end_epoch_utc   = int(end_sh.astimezone(pytz.UTC).timestamp())

# Restrict to db_* cmdb_ids present
db_mask = metric_df['cmdb_id'].astype(str).str.startswith('db_')
metric_db = metric_df[db_mask].copy()

# 1) Compute global P95 and total points per (cmdb_id, kpi_name) using full series
grouped_global = (
    metric_db
    .groupby(['cmdb_id','kpi_name'], sort=False)['value']
    .agg(total_points_full_series='count',
         global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter metric points to incident range and evaluate breaches by merging global thresholds
metric_in_range = metric_db[(metric_db['timestamp'] >= start_epoch_utc) & (metric_db['timestamp'] < end_epoch_utc)].copy()
metric_in_with_p95 = metric_in_range.merge(grouped_global, on=['cmdb_id','kpi_name'], how='left')

# Mark breaches: value > global_p95 (global_p95 computed on full series)
metric_in_with_p95['is_breach'] = metric_in_with_p95['value'] > metric_in_with_p95['global_p95']

# Aggregate per (cmdb_id, kpi_name)
in_range_agg = (
    metric_in_with_p95
    .groupby(['cmdb_id','kpi_name'], sort=False)
    .agg(
        in_range_point_count=('value','count'),
        in_range_max_value=('value','max'),
        in_range_breach_count=('is_breach','sum'),
        earliest_breach_ts=('timestamp', lambda s: s[metric_in_with_p95.loc[s.index,'is_breach']].min() if (metric_in_with_p95.loc[s.index,'is_breach'].any()) else pd.NA)
    )
    .reset_index()
)

# Merge aggregated in-range stats with global info to include KPIs that may have no in-range points
merged = grouped_global.merge(in_range_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts and set None where appropriate
merged['in_range_point_count'] = merged['in_range_point_count'].fillna(0).astype(int)
merged['in_range_breach_count'] = merged['in_range_breach_count'].fillna(0).astype(int)
merged['in_range_max_value'] = merged['in_range_max_value'].where(merged['in_range_max_value'].notna(), None)

# Convert earliest breach ts to Asia/Shanghai ISO or None
def ts_to_sh_iso(ts):
    if pd.isna(ts):
        return None
    try:
        dt_utc = datetime.fromtimestamp(int(ts), tz=timezone.utc)
        return dt_utc.astimezone(tz_sh).isoformat()
    except Exception:
        return None

merged['earliest_breach_timestamp_asia_shanghai'] = merged['earliest_breach_ts'].apply(ts_to_sh_iso)
merged.drop(columns=['earliest_breach_ts'], inplace=True)

# 3) Compute breach_severity_ratio for series that had at least one in-range breach
def compute_ratio(row):
    gp = row['global_p95']
    mv = row['in_range_max_value']
    if pd.isna(gp) or gp == 0 or mv is None:
        return None
    try:
        return float((mv - float(gp)) / float(gp))
    except Exception:
        return None

merged['breach_severity_ratio'] = merged.apply(compute_ratio, axis=1)

# Keep only KPIs with at least one in-range breach
breached = merged[merged['in_range_breach_count'] > 0].copy()

# 4) For each db, select KPI with largest breach_severity_ratio; omit dbs with no breached KPIs
# If ratio is None (e.g., gp==0), treat as -inf to avoid selecting; drop None ratios
breached_valid = breached[breached['breach_severity_ratio'].notna()].copy()

# If some dbs only have None ratios but have breaches (unlikely), include them by choosing highest raw (in_range_max - gp)
if breached_valid.empty and not breached.empty:
    # Fallback: compute ratio with special handling: gp==0 -> use mv-gp (absolute) as proxy
    breached['ratio_fallback'] = breached.apply(lambda r: (r['in_range_max_value'] - r['global_p95']) if r['in_range_max_value'] is not None else float('-inf'), axis=1)
    idx = breached.groupby('cmdb_id')['ratio_fallback'].idxmax()
    per_db = breached.loc[idx, :].copy()
else:
    idx = breached_valid.groupby('cmdb_id')['breach_severity_ratio'].idxmax()
    per_db = breached_valid.loc[idx, :].copy()

# Prepare final per-db rows with required fields
per_db = per_db.rename(columns={
    'kpi_name':'kpi_name_with_max_severity',
    'global_p95':'global_p95_value',
    'in_range_max_value':'in_range_max_value',
    'in_range_breach_count':'total_in_range_breaches_for_that_kpi',
    'earliest_breach_timestamp_asia_shanghai':'earliest_breach_timestamp_asia_shanghai'
})

# Select columns and sort by breach_severity_ratio desc, limit 10
final_cols = [
    'cmdb_id',
    'kpi_name_with_max_severity',
    'global_p95_value',
    'in_range_max_value',
    'breach_severity_ratio',
    'earliest_breach_timestamp_asia_shanghai',
    'total_in_range_breaches_for_that_kpi'
]
db_top = per_db[final_cols].sort_values(by='breach_severity_ratio', ascending=False).head(10).reset_index(drop=True)

# Small summary: overall db with largest severity and earliest breach timestamp across all dbs
if not db_top.empty:
    overall_row = db_top.iloc[0]
    overall_db_with_largest_severity = {'cmdb_id': overall_row['cmdb_id'], 'ratio': float(overall_row['breach_severity_ratio']) if overall_row['breach_severity_ratio'] is not None else None}
    # earliest breach timestamp across all breached series (use merged rows with breaches)
    timestamps = [r for r in merged['earliest_breach_timestamp_asia_shanghai'].tolist() if r is not None]
    earliest_breach_timestamp_across_all = min(timestamps) if timestamps else None
else:
    overall_db_with_largest_severity = None
    earliest_breach_timestamp_across_all = None

# Prepare JSON-serializable output
rows = []
for _, r in db_top.iterrows():
    rows.append({
        'cmdb_id': str(r['cmdb_id']),
        'kpi_name_with_max_severity': str(r['kpi_name_with_max_severity']),
        'global_p95_value': None if pd.isna(r['global_p95_value']) else float(r['global_p95_value']),
        'in_range_max_value': None if r['in_range_max_value'] is None else float(r['in_range_max_value']),
        'breach_severity_ratio': None if pd.isna(r['breach_severity_ratio']) else float(r['breach_severity_ratio']),
        'earliest_breach_timestamp_asia_shanghai': r['earliest_breach_timestamp_asia_shanghai'],
        'total_in_range_breaches_for_that_kpi': int(r['total_in_range_breaches_for_that_kpi'])
    })

result_db_severity = {
    'rows': rows,
    'summary': {
        'overall_db_with_largest_severity': overall_db_with_largest_severity,
        'earliest_breach_timestamp_across_all_dbs': earliest_breach_timestamp_across_all
    },
    'incident_range_asia_shanghai': {'start_iso': start_sh.isoformat(), 'end_iso': end_sh.isoformat()}
}

result_db_severity

```
Out[7]:
```
Incident window: 2020-05-23 00:30:00 – 01:00:00 (Asia/Shanghai)

Top DB-level severity findings (one KPI per DB with largest breach_severity_ratio)
1. db_002 — Physical_Read_Per_Sec
   - global P95: 0.205
   - in-range max: 16.6
   - breach_severity_ratio ≈ 79.98
   - earliest breach: 2020-05-23T00:48:00+08:00
   - in-range breaches: 2

2. db_010 — DFParaWrite_Per_Sec
   - global P95: 0.0705
   - in-range max: 1.82
   - breach_severity_ratio ≈ 24.82
   - earliest breach: 2020-05-23T00:42:00+08:00
   - in-range breaches: 1

3. db_001 — DFParaWrite_Per_Sec
   - global P95: 1.7505
   - in-range max: 42.25
   - breach_severity_ratio ≈ 23.14
   - earliest breach: 2020-05-23T00:30:00+08:00
   - in-range breaches: 2

4. db_003 — Session_pct
   - global P95: 1.51405
   - in-range max: 36.036
   - breach_severity_ratio ≈ 22.80
   - earliest breach: 2020-05-23T00:35:00+08:00
   - in-range breaches: 3

5. db_007 — Logic_Read_Per_Sec
   - global P95: 3,882.75
   - in-range max: 36,564.7
   - breach_severity_ratio ≈ 8.42
   - earliest breach: 2020-05-23T00:38:00+08:00
   - in-range breaches: 2

6. db_009 — Redo_Per_Sec
   - global P95: 23,871.125
   - in-range max: 55,250.7
   - breach_severity_ratio ≈ 1.31
   - earliest breach: 2020-05-23T00:56:00+08:00
   - in-range breaches: 2

7. db_011 — ACS
   - global P95: 0.01
   - in-range max: 0.02
   - breach_severity_ratio = 1.0
   - earliest breach: 2020-05-23T00:36:00+08:00
   - in-range breaches: 2

8. db_006 — Login_Per_Sec
   - global P95: 0.105
   - in-range max: 0.20
   - breach_severity_ratio ≈ 0.905
   - earliest breach: 2020-05-23T00:30:00+08:00
   - in-range breaches: 1

9. db_005 — Sess_Active
   - global P95: 0.705
   - in-range max: 1.3
   - breach_severity_ratio ≈ 0.844
   - earliest breach: 2020-05-23T00:38:00+08:00
   - in-range breaches: 1

10. db_004 — Redo_Per_Sec
    - global P95: 35,487.01
    - in-range max: 61,409.2
    - breach_severity_ratio ≈ 0.73
    - earliest breach: 2020-05-23T00:46:00+08:00
    - in-range breaches: 2

Small summary
- Overall DB with largest severity ratio: db_002 (ratio ≈ 79.98).
- Earliest breach timestamp across all DBs: 2020-05-23T00:30:00+08:00.
- Rows returned: top 10 DBs by severity ratio (as listed above).

Immediate recommendations
- Prioritize investigation of db_002 (extremely large ratio on Physical_Read_Per_Sec) and the other high-ratio DBs (db_010, db_001, db_003).  
- Start inspection around 00:30–00:40 for root-cause signals (query load, I/O issues, backups, or upstream spikes).

The original code execution output of IPython Kernel is also provided below for reference:

{'rows': [{'cmdb_id': 'db_002', 'kpi_name_with_max_severity': 'Physical_Read_Per_Sec', 'global_p95_value': 0.20499999999999974, 'in_range_max_value': 16.6, 'breach_severity_ratio': 79.97560975609768, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:48:00+08:00', 'total_in_range_breaches_for_that_kpi': 2}, {'cmdb_id': 'db_010', 'kpi_name_with_max_severity': 'DFParaWrite_Per_Sec', 'global_p95_value': 0.07049999999999998, 'in_range_max_value': 1.82, 'breach_severity_ratio': 24.815602836879442, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:42:00+08:00', 'total_in_range_breaches_for_that_kpi': 1}, {'cmdb_id': 'db_001', 'kpi_name_with_max_severity': 'DFParaWrite_Per_Sec', 'global_p95_value': 1.7505, 'in_range_max_value': 42.25, 'breach_severity_ratio': 23.13596115395601, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'total_in_range_breaches_for_that_kpi': 2}, {'cmdb_id': 'db_003', 'kpi_name_with_max_severity': 'Session_pct', 'global_p95_value': 1.51405, 'in_range_max_value': 36.036, 'breach_severity_ratio': 22.80106337307223, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00', 'total_in_range_breaches_for_that_kpi': 3}, {'cmdb_id': 'db_007', 'kpi_name_with_max_severity': 'Logic_Read_Per_Sec', 'global_p95_value': 3882.7499999999995, 'in_range_max_value': 36564.7, 'breach_severity_ratio': 8.417217178546133, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00', 'total_in_range_breaches_for_that_kpi': 2}, {'cmdb_id': 'db_009', 'kpi_name_with_max_severity': 'Redo_Per_Sec', 'global_p95_value': 23871.124999999993, 'in_range_max_value': 55250.7, 'breach_severity_ratio': 1.3145411035298928, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:56:00+08:00', 'total_in_range_breaches_for_that_kpi': 2}, {'cmdb_id': 'db_011', 'kpi_name_with_max_severity': 'ACS', 'global_p95_value': 0.01, 'in_range_max_value': 0.02, 'breach_severity_ratio': 1.0, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:36:00+08:00', 'total_in_range_breaches_for_that_kpi': 2}, {'cmdb_id': 'db_006', 'kpi_name_with_max_severity': 'Login_Per_Sec', 'global_p95_value': 0.10499999999999972, 'in_range_max_value': 0.2, 'breach_severity_ratio': 0.90476190476191, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'total_in_range_breaches_for_that_kpi': 1}, {'cmdb_id': 'db_005', 'kpi_name_with_max_severity': 'Sess_Active', 'global_p95_value': 0.7049999999999996, 'in_range_max_value': 1.3, 'breach_severity_ratio': 0.8439716312056748, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:38:00+08:00', 'total_in_range_breaches_for_that_kpi': 1}, {'cmdb_id': 'db_004', 'kpi_name_with_max_severity': 'Redo_Per_Sec', 'global_p95_value': 35487.01, 'in_range_max_value': 61409.2, 'breach_severity_ratio': 0.7304698254375331, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:46:00+08:00', 'total_in_range_breaches_for_that_kpi': 2}], 'summary': {'overall_db_with_largest_severity': {'cmdb_id': 'db_002', 'ratio': 79.97560975609768}, 'earliest_breach_timestamp_across_all_dbs': '2020-05-23T00:30:00+08:00'}, 'incident_range_asia_shanghai': {'start_iso': '2020-05-23T00:30:00+08:00', 'end_iso': '2020-05-23T01:00:00+08:00'}}```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
from datetime import datetime, timezone

# reuse metric_df if present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# ensure dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone and incident window
tz_sh = pytz.timezone('Asia/Shanghai')
start_sh = tz_sh.localize(datetime(2020,5,23,0,30,0))
end_sh   = tz_sh.localize(datetime(2020,5,23,1,0,0))
start_epoch_utc = int(start_sh.astimezone(pytz.UTC).timestamp())
end_epoch_utc   = int(end_sh.astimezone(pytz.UTC).timestamp())

# restrict to db_* cmdbs
metric_db = metric_df[metric_df['cmdb_id'].astype(str).str.startswith('db_')].copy()

# 1) global P95 and total points per (db, kpi) using full series
grouped_global = (
    metric_db
    .groupby(['cmdb_id','kpi_name'], sort=False)['value']
    .agg(total_points_full_series='count',
         global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) filter to incident window and evaluate breaches by merging global thresholds
metric_in_range = metric_db[(metric_db['timestamp'] >= start_epoch_utc) & (metric_db['timestamp'] < end_epoch_utc)].copy()
metric_in_with_p95 = metric_in_range.merge(grouped_global, on=['cmdb_id','kpi_name'], how='left')

# mark breaches
metric_in_with_p95['is_breach'] = metric_in_with_p95['value'] > metric_in_with_p95['global_p95']

# aggregate per (db, kpi)
in_range_agg = (
    metric_in_with_p95
    .groupby(['cmdb_id','kpi_name'], sort=False)
    .agg(
        in_range_point_count=('value','count'),
        in_range_max_value=('value','max'),
        in_range_breach_count=('is_breach','sum'),
        earliest_breach_ts_num=('timestamp', lambda s: s[metric_in_with_p95.loc[s.index,'is_breach']].min() if (metric_in_with_p95.loc[s.index,'is_breach'].any()) else pd.NA)
    )
    .reset_index()
)

# merge to get full per-kpi record
merged = grouped_global.merge(in_range_agg, on=['cmdb_id','kpi_name'], how='left')

# fill missing numeric counts
merged['in_range_point_count'] = merged['in_range_point_count'].fillna(0).astype(int)
merged['in_range_breach_count'] = merged['in_range_breach_count'].fillna(0).astype(int)
merged['in_range_max_value'] = merged['in_range_max_value'].where(merged['in_range_max_value'].notna(), None)

# compute breach_severity_ratio where applicable
def compute_ratio(row):
    gp = row['global_p95']
    mv = row['in_range_max_value']
    if pd.isna(gp) or gp == 0 or mv is None:
        return float('nan')
    return float((mv - float(gp)) / float(gp))

merged['breach_severity_ratio'] = merged.apply(compute_ratio, axis=1)

# keep only KPI series that had at least one in-range breach
kpis_breached = merged[merged['in_range_breach_count'] > 0].copy()

# helper to convert epoch -> Asia/Shanghai ISO
def epoch_to_sh_iso(ts):
    if pd.isna(ts):
        return None
    try:
        dt_utc = datetime.fromtimestamp(int(ts), tz=timezone.utc)
        return dt_utc.astimezone(tz_sh).isoformat()
    except Exception:
        return None

kpis_breached['earliest_breach_timestamp_asia_shanghai'] = kpis_breached['earliest_breach_ts_num'].apply(epoch_to_sh_iso)

# 2) Per-db aggregation
# number_of_kpi_series_with_breaches, total_in_range_breach_count, earliest_breach_timestamp_asia_shanghai
per_db_counts = (
    kpis_breached
    .groupby('cmdb_id', sort=False)
    .agg(
        number_of_kpi_series_with_breaches=('kpi_name','nunique'),
        total_in_range_breach_count=('in_range_breach_count','sum'),
        earliest_breach_ts_num=('earliest_breach_ts_num','min')
    )
    .reset_index()
)

# identify per-db KPI with max breach_severity_ratio
# drop NaN ratios so we pick meaningful ratios
kpis_valid_ratio = kpis_breached.dropna(subset=['breach_severity_ratio']).copy()
idx_max_ratio = kpis_valid_ratio.groupby('cmdb_id')['breach_severity_ratio'].idxmax()
per_db_max = kpis_valid_ratio.loc[idx_max_ratio, ['cmdb_id','kpi_name','global_p95','in_range_max_value','breach_severity_ratio','earliest_breach_timestamp_asia_shanghai','in_range_breach_count']].copy()
per_db_max = per_db_max.rename(columns={
    'kpi_name':'kpi_name_with_max_severity',
    'global_p95':'global_p95_value',
    'in_range_max_value':'in_range_max_value',
    'breach_severity_ratio':'max_breach_severity_ratio',
    'earliest_breach_timestamp_asia_shanghai':'earliest_breach_timestamp_of_max_severity_kpi',
    'in_range_breach_count':'total_in_range_breaches_for_max_kpi'
})

# join per_db_counts with per_db_max
per_db_summary = per_db_counts.merge(per_db_max, on='cmdb_id', how='left')

# convert earliest timestamps to ISO
per_db_summary['earliest_breach_timestamp_asia_shanghai'] = per_db_summary['earliest_breach_ts_num'].apply(epoch_to_sh_iso)
per_db_summary.drop(columns=['earliest_breach_ts_num'], inplace=True)

# finalize columns and sort by total_in_range_breach_count desc, limit 20
final_cols = [
    'cmdb_id',
    'number_of_kpi_series_with_breaches',
    'total_in_range_breach_count',
    'earliest_breach_timestamp_asia_shanghai',
    'kpi_name_with_max_severity',
    'max_breach_severity_ratio',
    'earliest_breach_timestamp_of_max_severity_kpi'
]
db_top20 = per_db_summary[final_cols].sort_values(by='total_in_range_breach_count', ascending=False).head(20).reset_index(drop=True)

# summary: overall db with most breaches and earliest breach across all dbs
if not per_db_summary.empty:
    overall_row = per_db_summary.loc[per_db_summary['total_in_range_breach_count'].idxmax()]
    overall_db_with_most_breaches = {'cmdb_id': overall_row['cmdb_id'], 'total_in_range_breach_count': int(overall_row['total_in_range_breach_count'])}
    # earliest across all breached KPIs
    earliest_all = kpis_breached['earliest_breach_ts_num'].min()
    earliest_breach_timestamp_across_all = epoch_to_sh_iso(earliest_all) if not pd.isna(earliest_all) else None
else:
    overall_db_with_most_breaches = None
    earliest_breach_timestamp_across_all = None

# prepare JSON-serializable rows
rows = []
for _, r in db_top20.iterrows():
    rows.append({
        'cmdb_id': str(r['cmdb_id']),
        'number_of_kpi_series_with_breaches': int(r['number_of_kpi_series_with_breaches']),
        'total_in_range_breach_count': int(r['total_in_range_breach_count']),
        'earliest_breach_timestamp_asia_shanghai': r['earliest_breach_timestamp_asia_shanghai'],
        'kpi_name_with_max_severity': None if pd.isna(r['kpi_name_with_max_severity']) else str(r['kpi_name_with_max_severity']),
        'max_breach_severity_ratio': None if pd.isna(r['max_breach_severity_ratio']) else float(r['max_breach_severity_ratio']),
        'earliest_breach_timestamp_of_max_severity_kpi': r['earliest_breach_timestamp_of_max_severity_kpi']
    })

result_db_aggregate = {
    'rows': rows,
    'summary': {
        'overall_db_with_most_breaches': overall_db_with_most_breaches,
        'earliest_breach_timestamp_across_all_dbs': earliest_breach_timestamp_across_all
    },
    'incident_range_asia_shanghai': {'start_iso': start_sh.isoformat(), 'end_iso': end_sh.isoformat()}
}

result_db_aggregate

```
Out[8]:
```
Incident window: 2020-05-23 00:30:00 – 01:00:00 (Asia/Shanghai)

Top DB-level findings (sorted by total in-range breach count)
- db_009: 23 KPI series breached, total_in_range_breach_count = 45  
  → KPI with max severity: Redo_Per_Sec (global P95 ≈ 23871.12, in-range max = 55250.7, severity ratio ≈ 1.31), earliest breach for DB: 2020-05-23T00:30:00+08:00, KPI earliest breach: 2020-05-23T00:56:00+08:00

- db_003: 23 KPI series breached, total_in_range_breach_count = 41  
  → KPI with max severity: Session_pct (global P95 ≈ 1.514, in-range max = 36.036, severity ratio ≈ 22.80), earliest breach: 2020-05-23T00:30:00+08:00, KPI earliest breach: 2020-05-23T00:35:00+08:00

- db_007: 21 KPI series breached, total_in_range_breach_count = 39  
  → KPI with max severity: Logic_Read_Per_Sec (global P95 ≈ 3,882.75, in-range max = 36,564.7, severity ratio ≈ 8.42), earliest breach: 2020-05-23T00:30:00+08:00, KPI earliest breach: 2020-05-23T00:38:00+08:00

- db_002: 19 KPI series breached, total_in_range_breach_count = 29  
  → KPI with max severity: Physical_Read_Per_Sec (global P95 ≈ 0.205, in-range max = 16.6, severity ratio ≈ 79.98), earliest breach: 2020-05-23T00:31:00+08:00, KPI earliest breach: 2020-05-23T00:48:00+08:00

- db_005: 18 KPI series breached, total_in_range_breach_count = 28  
  → KPI with max severity: Sess_Active (global P95 ≈ 0.705, in-range max = 1.3, severity ratio ≈ 0.844), KPI earliest breach: 2020-05-23T00:38:00+08:00

- db_012: 16 KPI series breached, total_in_range_breach_count = 26  
  → KPI with max severity: DFParaWrite_Per_Sec (severity ratio ≈ 0.488), KPI earliest breach: 2020-05-23T00:42:00+08:00

- db_006: 18 KPI series breached, total_in_range_breach_count = 25  
  → KPI with max severity: Login_Per_Sec (severity ratio ≈ 0.905), KPI earliest breach: 2020-05-23T00:30:00+08:00

- db_011: 13 KPI series breached, total_in_range_breach_count = 23  
  → KPI with max severity: ACS (global P95 = 0.01, in-range max = 0.02, severity ratio = 1.0), KPI earliest breach: 2020-05-23T00:36:00+08:00

- db_001: 14 KPI series breached, total_in_range_breach_count = 23  
  → KPI with max severity: DFParaWrite_Per_Sec (severity ratio ≈ 23.14), KPI earliest breach: 2020-05-23T00:30:00+08:00

- db_008: 14 KPI series breached, total_in_range_breach_count = 20  
  → KPI with max severity: DbTime (severity ratio ≈ 0.317), KPI earliest breach: 2020-05-23T00:58:00+08:00

Summary
- DB with the most total in-range breaches: db_009 (45 breaches).
- Earliest breach across all DBs: 2020-05-23T00:30:00+08:00.
- Notes: several DBs show both high counts of breached KPI series and high severity on individual KPIs. Particularly large severity ratios appear for:
  - db_002 (Physical_Read_Per_Sec, ratio ≈ 79.98),
  - db_010 and db_001 (DFParaWrite_Per_Sec, ratios ≈ 24.8 and 23.1),
  - db_003 (Session_pct, ratio ≈ 22.8),
  - db_007 (Logic_Read_Per_Sec, ratio ≈ 8.42).
  
Recommended next steps
- Prioritize investigation on DBs/KPIs with both high total breach counts (db_009, db_003, db_007) and extremely large severity ratios (db_002, db_001, db_003). Start around 00:30–00:40 and inspect raw series, query/load patterns, and I/O metrics for those KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

{'rows': [{'cmdb_id': 'db_009', 'number_of_kpi_series_with_breaches': 23, 'total_in_range_breach_count': 45, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'kpi_name_with_max_severity': 'Redo_Per_Sec', 'max_breach_severity_ratio': 1.3145411035298928, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:56:00+08:00'}, {'cmdb_id': 'db_003', 'number_of_kpi_series_with_breaches': 23, 'total_in_range_breach_count': 41, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'kpi_name_with_max_severity': 'Session_pct', 'max_breach_severity_ratio': 22.80106337307223, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:35:00+08:00'}, {'cmdb_id': 'db_007', 'number_of_kpi_series_with_breaches': 21, 'total_in_range_breach_count': 39, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'kpi_name_with_max_severity': 'Logic_Read_Per_Sec', 'max_breach_severity_ratio': 8.417217178546133, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_002', 'number_of_kpi_series_with_breaches': 19, 'total_in_range_breach_count': 29, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00', 'kpi_name_with_max_severity': 'Physical_Read_Per_Sec', 'max_breach_severity_ratio': 79.97560975609768, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:48:00+08:00'}, {'cmdb_id': 'db_005', 'number_of_kpi_series_with_breaches': 18, 'total_in_range_breach_count': 28, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00', 'kpi_name_with_max_severity': 'Sess_Active', 'max_breach_severity_ratio': 0.8439716312056748, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:38:00+08:00'}, {'cmdb_id': 'db_012', 'number_of_kpi_series_with_breaches': 16, 'total_in_range_breach_count': 26, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00', 'kpi_name_with_max_severity': 'DFParaWrite_Per_Sec', 'max_breach_severity_ratio': 0.4876033057851246, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:42:00+08:00'}, {'cmdb_id': 'db_006', 'number_of_kpi_series_with_breaches': 18, 'total_in_range_breach_count': 25, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'kpi_name_with_max_severity': 'Login_Per_Sec', 'max_breach_severity_ratio': 0.90476190476191, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_011', 'number_of_kpi_series_with_breaches': 13, 'total_in_range_breach_count': 23, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'kpi_name_with_max_severity': 'ACS', 'max_breach_severity_ratio': 1.0, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:36:00+08:00'}, {'cmdb_id': 'db_001', 'number_of_kpi_series_with_breaches': 14, 'total_in_range_breach_count': 23, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'kpi_name_with_max_severity': 'DFParaWrite_Per_Sec', 'max_breach_severity_ratio': 23.13596115395601, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:30:00+08:00'}, {'cmdb_id': 'db_008', 'number_of_kpi_series_with_breaches': 14, 'total_in_range_breach_count': 20, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:31:00+08:00', 'kpi_name_with_max_severity': 'DbTime', 'max_breach_severity_ratio': 0.31712015410546596, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:58:00+08:00'}, {'cmdb_id': 'db_004', 'number_of_kpi_series_with_breaches': 13, 'total_in_range_breach_count': 19, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:30:00+08:00', 'kpi_name_with_max_severity': 'Redo_Per_Sec', 'max_breach_severity_ratio': 0.7304698254375331, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:46:00+08:00'}, {'cmdb_id': 'db_010', 'number_of_kpi_series_with_breaches': 13, 'total_in_range_breach_count': 19, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:35:00+08:00', 'kpi_name_with_max_severity': 'DFParaWrite_Per_Sec', 'max_breach_severity_ratio': 24.815602836879442, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:42:00+08:00'}, {'cmdb_id': 'db_013', 'number_of_kpi_series_with_breaches': 13, 'total_in_range_breach_count': 17, 'earliest_breach_timestamp_asia_shanghai': '2020-05-23T00:33:00+08:00', 'kpi_name_with_max_severity': 'LFSync_Per_Sec', 'max_breach_severity_ratio': 0.6343490304709144, 'earliest_breach_timestamp_of_max_severity_kpi': '2020-05-23T00:35:00+08:00'}], 'summary': {'overall_db_with_most_breaches': {'cmdb_id': 'db_009', 'total_in_range_breach_count': 45}, 'earliest_breach_timestamp_across_all_dbs': '2020-05-23T00:30:00+08:00'}, 'incident_range_asia_shanghai': {'start_iso': '2020-05-23T00:30:00+08:00', 'end_iso': '2020-05-23T01:00:00+08:00'}}```
```